In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict

# Initialize the local Ollama model (llama3.2:1b running on your machine)
# No API key needed — this talks directly to your local Ollama server at localhost:11434
model = ChatOllama(
    model="llama3.2:1b",   # Model name (must be pulled via: ollama pull llama3.2:1b)
    temperature=0.7,        # Creativity level: 0 = deterministic, 1 = more creative
)

# Define the state schema using TypedDict
# This acts as a shared memory/context passed between nodes in the graph
class LLMState(TypedDict):
    question: str   # Input: the user's question
    answer: str     # Output: the model's response

# Define the graph node — this is the actual work unit
# It receives the current state, calls the LLM, and returns updated state
def llm_qa(state: LLMState):
    question = state["question"]  # Extract question from state

    # Invoke the local LLM with the question wrapped in a HumanMessage
    # HumanMessage tells the model this is a user turn (vs system/assistant)
    response = model.invoke([HumanMessage(content=question)])

    # Return updated state with the model's reply
    return {"question": question, "answer": response.content}

# ── Build the LangGraph workflow ──────────────────────────────────────────────

graph = StateGraph(LLMState)          # Create a graph that uses LLMState as its state

graph.add_node("llm_qa", llm_qa)     # Register the llm_qa function as a node

graph.add_edge(START, "llm_qa")      # Entry point: graph starts at llm_qa
graph.add_edge("llm_qa", END)        # Exit point: after llm_qa, graph ends

workflow = graph.compile()           # Compile the graph into a runnable workflow

# ── Run the workflow ──────────────────────────────────────────────────────────

# Provide the initial state (only question; answer is filled in by the node)
final_state = workflow.invoke({"question": "How far is the Moon from Earth?"})

# Print the answer returned by the LLM
print(final_state["answer"])

d:\AI\Ai Practice\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The average distance from the Earth to the Moon is approximately 238,855 miles (384,400 kilometers). This is often referred to as an "lunar distance." However, the Moon's orbit is not a perfect circle and its distance from Earth varies slightly over the course of a month. At its closest point, known as perigee, the Moon is about 225,622 miles (363,104 kilometers) away from Earth, and at its farthest point, known as apogee, it is about 252,088 miles (405,696 kilometers) away.
